In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit

from algorithm_evaluation import evaluate

# ---- LaTeX rendering for plots (Computer Modern, matches paper) ----
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}",
})

# The sweeps take hours. Leave False to load the shipped results from
# algorithm_evaluation/results/; set True to regenerate and overwrite them.
RECOMPUTE = False

## Simulation

In [ ]:
small_networks  = ['asia', 'sachs']
medium_networks = ['alarm', 'barley', 'child', 'insurance', 'mildew', 'water']
large_networks  = ['hailfinder', 'hepar2', 'win95pts']

all_graphs = (
    [(g, 'small-networks')  for g in small_networks]  +
    [(g, 'medium-networks') for g in medium_networks] +
    [(g, 'large-networks')  for g in large_networks]
)

path_prefix = 'algorithm_evaluation/example-graphs/bnlearn/'

# ── Config (matches ER evaluation) ─────────────────────────────────────────
theta_config = {
    'CF_fraction': 1.0,   # always a 2-world counterfactual query
    'W_sizes': [1],
    'Z_sizes': [1],
}

experiment_config = {
    'CF_fraction': 0.3,           # 30% counterfactual experiments
    'W_sizes': [1, 2, 3],
    'Z_sizes': [1, 2, 3],
}

N_SIMS   = 100 # 100
EXP_SIZE = 200 # 200
SEED     = 42

if RECOMPUTE:
    df = pd.DataFrame()

    for graph_name, subfolder in all_graphs:
        bif_path = f"{path_prefix}{subfolder}/{graph_name}.bif"
        print(f"Evaluating {graph_name}.bif ...")
        chunk = evaluate(
            graph_source="bnlearn",
            graph_config={"bif_path": bif_path},
            confounder_ratio_range=(0.1, 0.9),  
            confounder_strategy="random",    
            n_simulations=N_SIMS,
            seed=SEED,
            verbose=True,
            experiment_set_size=EXP_SIZE,
            theta_config=theta_config,
            experiment_config=experiment_config,
        )
        df = pd.concat([df, chunk], ignore_index=True)

    df.to_csv('algorithm_evaluation/results/bnlearn.csv', index=False)
    print(f"\nTotal simulations: {len(df)}")

## Summary statistics

In [ ]:
# Written above when RECOMPUTE is True; otherwise the shipped results.
df = pd.read_csv('algorithm_evaluation/results/bnlearn.csv')

frac_id = df['id_pruned'] / df['n_experiments']
frac_bl = df['blocked_pruned'] / df['n_experiments']

print("=== Overall ===")
print(f"Total simulations: {len(df)}")
print(f"Mean total pruning:         {df['fraction_pruned'].mean():.3f} (std {df['fraction_pruned'].std():.3f})")
print(f"Mean ID pruning:            {frac_id.mean():.3f} (std {frac_id.std():.3f})")
print(f"Mean interception pruning:  {frac_bl.mean():.3f} (std {frac_bl.std():.3f})")
print(f"Mean |U|/|V|:               {df['u_v_ratio'].mean():.3f} (std {df['u_v_ratio'].std():.3f})")
print(f"Max total pruning (sim):    {df['fraction_pruned'].max():.3f}")

# Per-graph interception (used for the min/max range in the paper)
per_graph_bl = (
    df.assign(frac_bl=frac_bl)
      .groupby('graph')['frac_bl']
      .agg(['mean', 'std'])
      .sort_values('mean')
)
per_graph_id = (
    df.assign(frac_id=frac_id)
      .groupby('graph')['frac_id']
      .agg(['mean', 'std'])
      .sort_values('mean')
)
per_graph_total = (
    df.groupby('graph')['fraction_pruned']
      .agg(['mean', 'std'])
      .sort_values('mean')
)
per_graph = (
  per_graph_bl.rename(columns={'mean': 'bl_mean', 'std': 'bl_std'})
  .join(per_graph_id.rename(columns={'mean': 'id_mean', 'std': 'id_std'}), how='outer')
  .join(per_graph_total.rename(columns={'mean': 'total_mean', 'std': 'total_std'}), how='outer')
  .round(3)
)

print("\nPer-graph summary (mean, std):")
print(per_graph.to_string())

## Per-graph results table

In [ ]:
summary = (
    df.assign(
        frac_id=df['id_pruned'] / df['n_experiments'],
        frac_bl=df['blocked_pruned'] / df['n_experiments'],
    )
    .groupby('graph')
    .agg(
        n_vars        = ('n_vars',          'first'),
        n_sims        = ('sim',             'count'),
        uv_mean       = ('u_v_ratio',       'mean'),
        uv_std        = ('u_v_ratio',       'std'),
        r_star_mean   = ('n_R_theta_star',  'mean'),
        id_mean       = ('frac_id',         'mean'),
        id_std        = ('frac_id',         'std'),
        bl_mean       = ('frac_bl',         'mean'),
        bl_std        = ('frac_bl',         'std'),
        total_mean    = ('fraction_pruned', 'mean'),
        total_std     = ('fraction_pruned', 'std'),
    )
    .sort_values('n_vars')
    .round(3)
)
print(summary.to_string())
print()
print(summary.to_latex(index=True))

## Paper plots

In [ ]:
# ---- tunable font sizes ----
LABEL_FONTSIZE  = 12
TICK_FONTSIZE   = 11
LEGEND_FONTSIZE = 11
HATCH_BLOCKED   = '///'   # B&W-readable pattern for AllPathsBlocked
# ----------------------------

plot_df = (
    df.assign(
        frac_id=df['id_pruned'] / df['n_experiments'],
        frac_bl=df['blocked_pruned'] / df['n_experiments'],
    )
    .groupby('graph', as_index=False)
    .agg(
        n_vars=('n_vars', 'first'),
        id_mean=('frac_id', 'mean'),
        bl_mean=('frac_bl', 'mean'),
        total_std=('fraction_pruned', 'std'),
    )
    .sort_values('n_vars')
)

fig, ax = plt.subplots(figsize=(12, 3.5))

x = np.arange(len(plot_df))
bar_w = 0.6

ax.bar(x, plot_df['bl_mean'], bar_w,
       color='#DD8452', alpha=0.88,
       edgecolor='black', linewidth=0.6, hatch=HATCH_BLOCKED,
       label='AllPathsBlocked')
ax.bar(x, plot_df['id_mean'], bar_w, bottom=plot_df['bl_mean'],
       color='#4C72B0', alpha=0.88,
       edgecolor='black', linewidth=0.6,
       label='ID check',
       yerr=plot_df['total_std'], capsize=3,
       error_kw=dict(ecolor='black', lw=1, alpha=0.7))

ax.set_xticks(x)
x_labels = [
    f"$\\texttt{{{g.replace('.BIF', '')}}}$\n$|\\mathbf{{V}}|$={n}"
    for g, n in zip(plot_df['graph'].str.upper(), plot_df['n_vars'])
]
ax.set_xticklabels(
    x_labels,
    rotation=30, ha='right', fontsize=TICK_FONTSIZE
)
ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
ax.set_ylabel('Fraction of experiments pruned', fontsize=LABEL_FONTSIZE)
# ax.set_title(
#     r'bnlearn graphs: pruning breakdown  '
#     r'($|U|/|V| \\approx 0.20$, non-ancestor confounders, CF$_\\mathrm{exp}$=0.3)',
#     fontsize=10
# )
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
# ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.9)

# Annotate total pruning above each bar (offset above the error bar)
# for i, row in enumerate(plot_df.itertuples()):
#     total = row.id_mean + row.bl_mean
#     ax.text(i, min(total + row.total_std + 0.01, 0.99),
#             f'{total:.2f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('plots/bnlearn_per_graph.png',
            bbox_inches='tight', dpi=300)
plt.show()
print(df.shape[0], "total simulations")

## Pruning breakdown vs $|R^*_\theta|/|V|$

In [ ]:
# ---- tunable font sizes ----
LABEL_FONTSIZE  = 13
TICK_FONTSIZE   = 12
LEGEND_FONTSIZE = 12
LINESTYLE_BL    = '--'    # dashed line for B&W readability
HATCH_BL        = '///'   # hatch on the std band
# ----------------------------

sub_bl = df.assign(
    r_star_frac  = df['n_R_theta_star'] / df['n_vars'],
    frac_blocked = df['blocked_pruned'] / df['n_experiments'],
).copy()

r_max  = sub_bl['r_star_frac'].max()
bins   = np.linspace(0, min(r_max * 1.05, 0.90), 18)
sub_bl['r_mid'] = pd.cut(sub_bl['r_star_frac'], bins=bins).apply(
    lambda x: x.mid if pd.notna(x) else np.nan
)

grp_bl = (
    sub_bl.groupby('r_mid', as_index=False)
    .agg(mean=('frac_blocked', 'mean'), std=('frac_blocked', 'std'))
    .dropna()
    .sort_values('r_mid')
)
grp_bl['r_mid'] = grp_bl['r_mid'].astype(float)

plt.figure(figsize=(8, 5))
plt.plot(grp_bl['r_mid'], grp_bl['mean'], color='#DD8452', lw=2,
         linestyle=LINESTYLE_BL,
         label='Pruned by interception (mean)')
plt.fill_between(grp_bl['r_mid'],
                 (grp_bl['mean'] - grp_bl['std']).clip(0),
                 (grp_bl['mean'] + grp_bl['std']).clip(0, 1),
                 alpha=0.15, color='#DD8452', hatch=HATCH_BL, edgecolor='#DD8452',
                 label=r'$\pm$ 1 std')

plt.xlabel(r'$|\mathbf{R}^*_\theta|/|\mathbf{V}|$', fontsize=LABEL_FONTSIZE)
plt.ylabel('Fraction of experiments', fontsize=LABEL_FONTSIZE)
plt.xticks(fontsize=TICK_FONTSIZE)
plt.yticks(fontsize=TICK_FONTSIZE)
# plt.title(r'Pruning by interception vs. $|R^*_\theta|/|V|$ (bnlearn graphs)', fontsize=11)
plt.ylim(0, 0.8)
plt.grid(alpha=0.3)
plt.legend(fontsize=LEGEND_FONTSIZE)
plt.tight_layout()
plt.savefig('plots/bnlearn_against_Rstartheta.png',
            bbox_inches='tight', dpi=300)
plt.show()